# Advanced Certification Programme in Agentic and Generative AI
## A Programme by IISc and TalentSprint
### Mini-Project 1 Part-B: LLM Based Spam Classification and Gradio Interface

**(Solution)**

**DISCLAIMER:** THIS NOTEBOOK IS PROVIDED ONLY AS A REFERENCE SOLUTION NOTEBOOK FOR THE MINI-PROJECT. THERE MAY BE OTHER POSSIBLE APPROACHES/METHODS TO ACHIEVE THE SAME RESULTS.

## Learning Objectives

At the end of the mini-project, you will be able to :

* Load the trained NN model from Part-A
* Initialize models like `gpt-oss-20b`, `llama-3.1-8b-instant` to use via Groq API key (free-tier)
* Run Ollama server and initialize models like `gemma3:4b`, `llama3.1:8b`
* Experiment with different prompting techniques for spam classification
* Create a Gradio interface to enable users to choose the model to use for response generation


## Problem Statement

Spam messages continue to pose a significant challenge across email and messaging platforms, often leading to security risks, phishing attempts, and unwanted content. Detecting and filtering spam effectively requires robust classification techniques that can adapt to evolving message patterns.

Traditional machine learning models can perform spam detection based on learned patterns in historical data, while modern LLMs offer alternative approaches through prompt-based reasoning and text understanding.

In this project, the objective is to build **a spam classification system** that uses both **a trained neural network model and multiple Large Language Models (LLMs)**.

The project begins by loading the neural network model developed in Part-A as one of the spam classifier. In addition, learners have to configure and use LLMs accessed through the **Groq API** as well as locally hosted models running through **Ollama**, enabling experimentation with different inference environments.

You will also explore **prompt engineering techniques**, including zero-shot and few-shot prompting, to perform spam classification using LLMs and compare their performance with the neural network baseline.

To make the system interactive and user-friendly, a **Gradio-based interface** will be developed that allows users to input a message and select which model (neural network, Groq-hosted LLM, or locally hosted Ollama model) should be used for classification.

The final system will demonstrate how different AI approaches—traditional machine learning and modern LLM-based inference—can be used and evaluated within a single application for solving a real-world **spam detection problem**.

## Dataset

(Used for training the Neural Network model)

The [SMS Spam Collection dataset](https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset) is a set of SMS tagged messages that have been collected for SMS Spam research. It contains one set of SMS messages in English of 5,574 messages, tagged acording being ham (legitimate) or spam.


## Grading = 6 Points

In [ ]:
# prompt: Create a hidden code cell with @#title Download the Dataset. Data should be downloaded from the following link: https://cdn.exec.talentsprint.com/static/aimlops/c3/spam.csv

#@title Download the Dataset
!wget https://cdn.exec.talentsprint.com/static/aimlops/c3/spam.csv


### Import Neccesary Packages

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import re
import pickle
from tensorflow.keras.layers import TextVectorization

### Load the Trained Neural Network Model from Part-A

Load the neural network model that was trained and saved in Part-A of the project.

This model will be used as one of the spam classifiers to generate predictions and compare its performance with LLM-based approaches.

In [ ]:
# First, execute the Part-A solution notebook, then download the below files from there
# - `spam_classifier_state_dict.pth`
# - `spam_classifier_scripted.pt`
# - `vectorizer_vocab.pkl`

# Upload the above files in this notebook, then execute the below code cells

In [ ]:
import torch
import torch.nn as nn

# Define the SpamClassifier class again, as it's needed to load the state_dict
# This definition should be identical to the one used during training.
class SpamClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super(SpamClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, text):
        embedded = self.embedding(text)
        output, (hidden, cell) = self.lstm(embedded)
        final_output = self.fc(hidden[-1, :, :])
        return final_output

# Model parameters (must match the parameters used to create the saved model)
vocab_size = 5000
embedding_dim = 128
hidden_dim = 256
output_dim = 1

# Option 1: Load the model's state dictionary
loaded_model_state_dict = SpamClassifier(vocab_size, embedding_dim, hidden_dim, output_dim)
loaded_model_state_dict.load_state_dict(torch.load('spam_classifier_state_dict.pth'))
loaded_model_state_dict.eval() # Set the model to evaluation mode
print("Model state dictionary loaded successfully!")

# Option 2: Load the entire scripted model
loaded_scripted_model = torch.jit.load('spam_classifier_scripted.pt')
loaded_scripted_model.eval() # Set the model to evaluation mode
print("Scripted model loaded successfully!")


Model state dictionary loaded successfully!
Scripted model loaded successfully!


In [ ]:
import re

def clean_text(text):
  """
  This function cleans the text by removing punctuation, converting to lowercase,
  and removing extra whitespace.
  """
  text = text.lower()  # Convert to lowercase
  text = re.sub(r'[^\w\s]', '', text)  # Remove punctuation
  text = " ".join(text.split())  # Remove extra whitespace
  return text


In [ ]:
import pickle
from tensorflow.keras.layers import TextVectorization

# Load vocabulary
with open("vectorizer_vocab.pkl", "rb") as f:
    vocab = pickle.load(f)

# Recreate vectorizer
vectorize_layer = TextVectorization(
    max_tokens=5000,
    output_mode='int',
    output_sequence_length=100
)

# Set the saved vocabulary
vectorize_layer.set_vocabulary(vocab)

In [ ]:
new_text = "Thanks for sending the notes at this time."

# 1. Clean the text
cleaned_text = clean_text(new_text)

# 2. Vectorize the text
vectorized_text = vectorize_layer([cleaned_text])

# 3. Convert to PyTorch tensor
input_tensor = torch.tensor(vectorized_text.numpy(), dtype=torch.long)

# 4. Pass the tensor to the model
output = loaded_scripted_model(input_tensor)

print("Model output (logits):")
print(output)

# To get a probability and predicted class (spam/ham):
probability = torch.sigmoid(output).item()
predicted_class = "Spam" if probability > 0.5 else "Ham"

print(f"\nInput text: '{new_text}'")
print(f"Predicted probability: {probability:.4f}")
print(f"Predicted class: {predicted_class}")


Model output (logits):
tensor([[-1.9701]], grad_fn=<AddmmBackward0>)

Input text: 'Thanks for sending the notes at this time.'
Predicted probability: 0.1224
Predicted class: Ham


In [ ]:
def get_prediction_nn_model(text):
    # Clean the text
    cleaned_text = clean_text(new_text)

    # Vectorize the text
    vectorized_text = vectorize_layer([cleaned_text])

    # Convert to PyTorch tensor
    input_tensor = torch.tensor(vectorized_text.numpy(), dtype=torch.long)

    # Pass the tensor to the model
    output = loaded_scripted_model(input_tensor)

    # To get a probability and predicted class (spam/ham):
    probability = torch.sigmoid(output).item()
    predicted_class = "Spam" if probability > 0.5 else "Ham"

    return predicted_class


In [ ]:
get_prediction_nn_model("Thanks for sending the notes at this time.")

'Ham'

### Initialize Models such as `gpt-oss-20b` and `llama-3.1-8b-instant` using the **Groq** API Key (Free Tier)

Configure access to the Groq API using the your Groq API key and initialize the mentioned large language models.

These models will be used to perform spam classification by sending prompts through the Groq inference API.

**Hint:** Refer to the *Supplementary Notebook - Getting Started with Groq API and Ollama Server.ipynb*, released along with this notebook.

In [ ]:
# Save the key in Colab's Secrets then load from there

import os
from google.colab import userdata

os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')

In [ ]:
# Install Groq library
!pip -q install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 4.9 MB/s eta 0:00:00


In [ ]:
from groq import Groq

# Initialize Groq client
groq_client = Groq()

In [ ]:
prompt_template = """
You are an AI system designed to detect spam messages.
A message is considered Spam if it contains unsolicited advertisements, suspicious links, prizes, scams, or requests for personal information.

Examples:

Message: "Win a free iPhone now! Click here to claim your reward."
Label: Spam

Message: "Hey, are we still meeting for lunch tomorrow?"
Label: Ham

Message: "Limited time offer! Get 50% discount on all products today."
Label: Spam

Now classify the message below as Spam or Ham. Answer only with one word.

Message: "{message}"
Label:
"""

In [ ]:
message = "Win a free iPhone now! Click here to claim your reward."

formatted_prompt = prompt_template.format(message=message)
print(formatted_prompt)


You are an AI system designed to detect spam messages.
A message is considered Spam if it contains unsolicited advertisements, suspicious links, prizes, scams, or requests for personal information.

Examples:

Message: "Win a free iPhone now! Click here to claim your reward."
Label: Spam

Message: "Hey, are we still meeting for lunch tomorrow?"
Label: Ham

Message: "Limited time offer! Get 50% discount on all products today."
Label: Spam

Now classify the message below as Spam or Ham. Answer only with one word.

Message: "Win a free iPhone now! Click here to claim your reward."
Label:



In [ ]:
# Using `gpt-oss-20b` model

# Send the prompt to the Groq Chat Completion API
# This creates a request to generate a response from the LLM

response = groq_client.chat.completions.create(
    model="openai/gpt-oss-20b",                          # Specify the LLM to be used
    messages=[                                            # Provide the conversation as a list of messages
        {"role": "user", "content": formatted_prompt}               # Here, we send only one user message containing the prompt
    ]
)

print(response.choices[0].message.content)

Spam


In [ ]:
# Using `llama-3.1-8b-instant` model

# Send the prompt to the Groq Chat Completion API
# This creates a request to generate a response from the LLM

response = groq_client.chat.completions.create(
    model="llama-3.1-8b-instant",                          # Specify the LLM to be used
    messages=[                                            # Provide the conversation as a list of messages
        {"role": "user", "content": formatted_prompt}               # Here, we send only one user message containing the prompt
    ]
)

print(response.choices[0].message.content)

Spam


In [ ]:
def get_response_groq_model(message, model_name):
    formatted_prompt = prompt_template.format(message=message)
    response = groq_client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "user", "content": formatted_prompt}
        ]
    )
    return response.choices[0].message.content


In [ ]:
get_response_groq_model("Win a free iPhone now! Click here to claim your reward.", "openai/gpt-oss-20b")

'Spam'

In [ ]:
get_response_groq_model("Win a free iPhone now! Click here to claim your reward.", "llama-3.1-8b-instant")

'Spam'

### Run the **Ollama Server** and Initialize Models such as `gemma3:4b` and `llama3.1:8b`

Start the Ollama server locally in COlab and load the required open-source models.

These models will allow local inference for spam classification, enabling experimentation without relying on external APIs.

**Hint:** Refer to the *Supplementary Notebook - Getting Started with Groq API and Ollama Server.ipynb*, released along with this notebook.

In [ ]:
# Install the 'zstd' (Zstandard) compression utility using the system package manager.
# This tool is commonly required for extracting or handling .zst compressed files.

!sudo apt-get install zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 2 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (1,092 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 117540 files and directories currently 

In [ ]:
# Download and execute the official Ollama installation script.
# - `curl -fsSL` fetches the script silently and securely from the URL.
# - The pipe `| sh` passes the downloaded script directly to the shell for execution.
# This installs Ollama on the system.

!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


**You may ignore the above WARNING message.**

In [ ]:
# Start the Ollama server as a background process.
# - `nohup` ensures the process continues running even if the session disconnects.
# - `> ollama.log` redirects standard output to a log file.
# - `2>&1` redirects error messages to the same log file.
# - `&` runs the command in the background.

!nohup ollama serve > ollama.log 2>&1 &

# Rerun this command in case Ollama shows "Connection refused" error

In [ ]:
# List all running processes and filter for those related to Ollama.
# - `ps aux` displays all active processes with detailed information.
# - `| grep ollama` filters the output to show only Ollama-related processes.

!ps aux | grep ollama

root       24463  0.6  0.2 1862116 35356 ?       Sl   11:55   0:00 ollama serve
root       24508  0.0  0.0   7372  3452 ?        S    11:55   0:00 /bin/bash -c ps aux | grep ollama
root       24510  0.0  0.0   6480  2428 ?        S    11:55   0:00 grep ollama


In [ ]:
# Download the "gemma3:4b" model using Ollama.
# - `%%bash` runs the entire cell in a bash shell instead of Python.
# This allows real-time streaming of the download progress.

%%bash
ollama pull gemma3:4b

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest 
pulling aeda25e63ebd:   0% ▕                  ▏  59 KB/3.3 GB                  pulling manifest 
pulling aeda25e63ebd:   1% ▕                  ▏  30 MB/3.3 GB                  pulling manifest 
pulling aeda25e63ebd:   2% ▕                  ▏  53 MB/3.3 GB                  pulling manifest 
pulling aeda25e63ebd:   2% ▕                  ▏  67 MB/3.3 GB                  pulling manifest 
pulling aeda25e63ebd:   3% ▕                  ▏  90 MB/3.3 GB                  pulling manifest 
pulling aeda25e63ebd:   3% ▕                  ▏ 111 MB/3.3 GB                  pulling manifest 
pulling aeda25e63ebd:   4% ▕                  ▏ 120 MB/3.3 GB                  pulling manifest 
pulling aeda25e63ebd:   4% ▕                  ▏ 137 MB/3.3 GB                  pulling manifest 
pulling aeda25e6

In [ ]:
# Download the "llama3.1:8b" model using Ollama.

%%bash
ollama pull llama3.1:8b

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest 
pulling 667b0c1932bc:   0% ▕                  ▏ 8.0 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:   1% ▕                  ▏  45 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:   1% ▕                  ▏  71 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:   2% ▕                  ▏  96 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:   2% ▕                  ▏ 114 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:   3% ▕                  ▏ 135 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:   3% ▕                  ▏ 160 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:   4% ▕                  ▏ 176 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:   4% ▕                  ▏ 208 MB

In [ ]:
# Install the `langchain-ollama` package quietly
# - `-q` suppresses detailed installation logs
# This package enables integration between LangChain and Ollama models

!pip -q install langchain-ollama

In [ ]:
# Import the ChatOllama class from the langchain_ollama package
# This class allows interaction with Ollama-hosted LLMs using LangChain

from langchain_ollama import ChatOllama

In [ ]:
# Initialize the LLM (Large Language Model) instance
gemma_llm = ChatOllama(
    model = "gemma3:4b",                   # Specify the model name available in your Ollama instance
    base_url="http://127.0.0.1:11434",     # <---- change this as per your Codespace address
    validate_model_on_init = True,
    temperature = 0,                     # Controls randomness of the output
    num_predict = 15,                     # Maximum number of tokens the model is allowed to generate
)

In [ ]:
# Initialize the LLM (Large Language Model) instance
llama_llm = ChatOllama(
    model = "llama3.1:8b",                   # Specify the model name available in your Ollama instance
    base_url="http://127.0.0.1:11434",     # <---- change this as per your Codespace address
    validate_model_on_init = True,
    temperature = 0,                     # Controls randomness of the output
    num_predict = 15,                     # Maximum number of tokens the model is allowed to generate
)

In [ ]:
output = gemma_llm.invoke("What is 2 plus 3")
print(output.content)

2 + 3 = 5



### Experiment with Different Prompting Techniques for Spam Classification

Design and test various prompting strategies such as ***zero-shot prompting***, ***few-shot prompting***, and ***instruction-based prompts***.

Evaluate how different prompt structures influence the accuracy and reliability of spam classification using LLMs.

In [ ]:
prompt_template = """
You are an AI system designed to detect spam messages.
A message is considered Spam if it contains unsolicited advertisements, suspicious links, prizes, scams, or requests for personal information.

Examples:

Message: "Win a free iPhone now! Click here to claim your reward."
Label: Spam

Message: "Hey, are we still meeting for lunch tomorrow?"
Label: Ham

Message: "Limited time offer! Get 50% discount on all products today."
Label: Spam

Now classify the message below as Spam or Ham. Answer only with one word.

Message: "{message}"
Label:
"""

In [ ]:
message = "Win a free iPhone now! Click here to claim your reward."

formatted_prompt = prompt_template.format(message=message)
print(formatted_prompt)


You are an AI system designed to detect spam messages.
A message is considered Spam if it contains unsolicited advertisements, suspicious links, prizes, scams, or requests for personal information.

Examples:

Message: "Win a free iPhone now! Click here to claim your reward."
Label: Spam

Message: "Hey, are we still meeting for lunch tomorrow?"
Label: Ham

Message: "Limited time offer! Get 50% discount on all products today."
Label: Spam

Now classify the message below as Spam or Ham. Answer only with one word.

Message: "Win a free iPhone now! Click here to claim your reward."
Label:



In [ ]:
output = gemma_llm.invoke(formatted_prompt)
print(output.content)

Spam



In [ ]:
output = llama_llm.invoke(formatted_prompt)
print(output.content)

Spam


In [ ]:
def get_response_ollama_model(message, model_name):

    formatted_prompt = prompt_template.format(message=message)

    if model_name == "gemma3:4b":
        output = gemma_llm.invoke(formatted_prompt)
        return output.content

    elif model_name == "llama3.1:8b":
        output = llama_llm.invoke(formatted_prompt)
        return output.content


In [ ]:
get_response_ollama_model("Win a free iPhone now! Click here to claim your reward.", "gemma3:4b")

'Spam\n'

### Create a `Gradio Interface` to Allow Users to Select the Model for Response Generation

Develop an interactive **Gradio interface** (as shown below) where users can input a message and select which model to use for classification.

The interface should support switching between models (e.g., Groq-hosted models, Ollama models, or the trained neural network) and display the predicted result to the user along with the time it took to get the response.

<img src='https://drive.google.com/uc?id=1qgT3Zp08gMwY9a_UT8_kiOG2BdT1Qmdm' width=800px>

In [ ]:
import gradio
import gradio as gr

In [ ]:
# Input Elements
msg_in = gr.Textbox(label="Message")
model_in = gr.Radio(
    label="Model",
    choices=["Neural Network Model",
             "Groq model: gpt-oss-20b",
             "Groq model: llama-3.1-8b-instant",
             "Ollama model: gemma3:4b",
             "Ollama model: llama3.1:8b"],
    value="Neural Network Model",
    type="value"
)

# Output Elements
label_out = gr.Textbox(label="Prediction")
latency_out = gr.Textbox(label="Latency (sec)")

In [ ]:
import time

def get_prediction(message, model_name):
    if model_name == "Neural Network Model":
        start_time = time.time()
        prediction = get_prediction_nn_model(message)
        end_time = time.time()
        time_taken = end_time - start_time
        return prediction, round(time_taken, 4)

    elif model_name == "Groq model: gpt-oss-20b":
        start_time = time.time()
        prediction = get_response_groq_model(message, "openai/gpt-oss-20b")
        end_time = time.time()
        time_taken = end_time - start_time
        return prediction, round(time_taken, 4)

    elif model_name == "Groq model: llama-3.1-8b-instant":
        start_time = time.time()
        prediction = get_response_groq_model(message, "llama-3.1-8b-instant")
        end_time = time.time()
        time_taken = end_time - start_time
        return prediction, round(time_taken, 4)

    elif model_name == "Ollama model: gemma3:4b":
        start_time = time.time()
        prediction = get_response_ollama_model(message, "gemma3:4b")
        end_time = time.time()
        time_taken = end_time - start_time
        return prediction, round(time_taken, 4)

    elif model_name == "Ollama model: llama3.1:8b":
        start_time = time.time()
        prediction = get_response_ollama_model(message, "llama3.1:8b")
        end_time = time.time()
        time_taken = end_time - start_time
        return prediction, round(time_taken, 4)


In [ ]:
get_prediction(formatted_prompt, "Groq model: llama-3.1-8b-instant")

('Spam', 0.1882)

In [ ]:
get_prediction(formatted_prompt, "Ollama model: gemma3:4b")

('Spam\n', 78.0452)

In [ ]:
# Create Interface
iface = gr.Interface(
    fn=get_prediction,
    inputs=[msg_in, model_in],
    outputs=[label_out, latency_out],
    title="Spam Classification",
    description="Enter a message and select a model to classify it as Spam or Ham.",
    flagging_mode="never",
)

In [ ]:
iface.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://4f3efdf873afd7f7da.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://4f3efdf873afd7f7da.gradio.live


---

<center>
$END$
</center>

---